# Switch Administration & Security — Revision Notes

SwitchLab lessons 9–15: securing access to a Cisco switch.

**Not repeated here (already in [ios_commands_1](#notebook-786572401935295)):** lesson 9 — save config (`write memory` / `copy run start`), lesson 10 — investigate first (`show` commands). Only referenced where they fit.

**Flow:** login banner → device passwords → password encryption → remote access (VTY) → user-based login → the full hardening sequence → interview Q&A → summary

**Companions:** [ios_commands_1](#notebook-786572401935295) (basic commands) · [networking_fundamentals_1](#notebook-4479634647434063) (concepts)

## 1. Login Banner — `banner motd`

* MOTD = Message Of The Day — shown to anyone who connects, **before** login

`banner motd # Authorized Access Only #`

* Text goes between the `#` delimiters
* Uses: legal warning, company notice, security reminder
* Interview line: a banner is a **warning, not a lock** — it doesn't block access; passwords do

## 2. Device Passwords — Two Doors to Protect

| Password | Protects | Configuration |
| --- | --- | --- |
| Console password | physical access (console cable) | `line console 0` → `password ...` → `login` |
| Enable password | privileged EXEC (admin `#` mode) | `enable password ...` (global config) |

* Console = the port on the switch you physically plug into
* Enable = the mode where configuration changes happen
* Purpose: restrict unauthorized access to the device and its configuration

## 3. Hide Passwords — `service password-encryption`

* Without it: `show running-config` displays passwords in **plain text**
* With it: `password 7 xxxxxxxxx` — masked in the config output
* One global command, masks all line/console passwords

**Interview caveat:** type-7 is weak, reversible obfuscation — for privileged mode use `enable secret <password>` (hashed, stronger) instead of `enable password`

## 4. Remote Access — VTY Lines

**What a "line" is**

* A line = one login session into the switch
* Console = 1 physical line (front door — you must physically be there)
* VTY (Virtual TeletYpe) = **virtual** lines — no cable, sessions arriving over the network (Telnet/SSH)

**The logic**

* `line vty 0 4` = configure remote sessions 0, 1, 2, 3, 4 → 5 lines = up to 5 simultaneous remote logins
* Someone connects remotely → the switch checks that line's rules: `password` + `login` = ask for this password first
* No VTY password set → Cisco refuses remote logins
* `login local` (next section) = check usernames instead of the shared password

**Think:** console = front door; VTY = 5 phone lines anyone in the world can call — so every call must answer "who's this?"

```
line vty 0 4
password Remote123
login
```

**Telnet vs SSH — how remote users actually connect**

| | Telnet | SSH |
| --- | --- | --- |
| Purpose | remote CLI login | remote CLI login |
| Encryption | none — passwords in plain text | encrypted |
| Port | 23 | 22 |

* Both arrive through VTY lines — that's why the VTY password protects them
* Interview line: Telnet is the old way, SSH its secure replacement — Telnet should never be used on untrusted networks

## 5. User-Based Login — `login local`

```
username admin privilege 15 password admin123
line vty 0 4
login local
```

* `username ... password ...` — creates a local user database on the switch
* `privilege 15` — full administrator level (privilege 0 = basic User EXEC)
* `login local` — authenticate against local users **instead of one shared password**

**Why better:** per-user accountability — you know exactly WHO logged in; a shared password never tells you

## 6. Securing a New Switch — the Full Sequence

1. `banner motd` — warn unauthorized users
2. Console password — protect physical access
3. `enable secret` — protect privileged mode
4. `service password-encryption` — hide passwords from `show run`
5. VTY password + `login` — protect remote access
6. `login local` + usernames — per-user accountability
7. `write memory` — save it all (→ ios_commands_1)

* Interview scenario: "How do you secure a fresh switch?" → walk this list in order — that ordering alone tells the story

## 7. Interview Q&A

**One-liners (say-aloud)**

| Command / term | One-liner |
| --- | --- |
| `banner motd` | message shown before login — warning/notice |
| Console password | protects the physical console port |
| `enable password` / `enable secret` | protects privileged mode; secret is hashed, always prefer it |
| `service password-encryption` | masks passwords in the config (weak type-7) |
| VTY | virtual terminal lines — remote Telnet/SSH sessions |
| Telnet / SSH | remote CLI login — port 23 / 22; SSH encrypts, Telnet is plain text |
| `login` | require the line password |
| `login local` | require a local username + password |
| `privilege 15` | full administrator level |

**Scenario drills**

* Can't Telnet in, cable and IP are fine? → no VTY password set (or `login` missing)
* Passwords visible in `show running-config`? → `service password-encryption` not applied
* Audit needs to know who logged in? → switch from shared password to `login local` with usernames

**Red flags to avoid**

* Confusing console / enable / VTY passwords
* Thinking `banner motd` blocks access
* Recommending `enable password` where `enable secret` exists

## 8. Key Concepts Summary

| Concept | One-liner |
| --- | --- |
| `banner motd` | pre-login warning message |
| Console password | the physical-access door |
| `enable secret` | the privileged-mode door, hashed |
| `service password-encryption` | type-7 masking in `show run` |
| `line vty 0 4` | the first 5 remote session lines |
| Telnet vs SSH | port 23 vs 22 — SSH encrypts, Telnet plain text |
| `login` | require the line password |
| `login local` | require local username + password |
| `privilege 15` | full admin level |
| Hardening order | banner → console → enable → encrypt → VTY → local users → save |